Token classification assigns a label to individual tokens in a sentence. One of the most common token classification tasks is Named Entity Recognition (NER). NER attempts to find a label for each entity in a sentence, such as a person, location, or organization.

This guide will show you how to:

1. Finetune [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) on the [WNUT 17](https://huggingface.co/datasets/wnut_17) dataset to detect new entities.
2. Use your finetuned model for inference.

In [ ]:
! pip install transformers datasets evaluate accelerate huggingface_hub

In [ ]:
user_name = "amin-oj"
from huggingface_hub import notebook_login
notebook_login()

## Load WNUT 17 dataset

In [ ]:
!pip install 'datasets<4.0.0'
# https://discuss.huggingface.co/t/dataset-scripts-are-no-longer-supported/163891

In [ ]:
from datasets import load_dataset
wnut = load_dataset("wnut_17")
wnut["train"][0]

Each number in `ner_tags` represents an entity. Convert the numbers to their label names to find out what the entities are:

In [ ]:
label_list = wnut["train"].features[f"ner_tags"].feature.names
label_list

The letter that prefixes each `ner_tag` indicates the token position of the entity:

- `B-` indicates the beginning of an entity.
- `I-` indicates a token is contained inside the same entity (for example, the `State` token is a part of an entity like
  `Empire State Building`).
- `0` indicates the token doesn't correspond to any entity.

## Preprocess

In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

As you saw in the example `tokens` field above, it looks like the input has already been tokenized. But the input actually hasn't been tokenized yet and you'll need to set `is_split_into_words=True` to tokenize the words into subwords. For example:

In [ ]:
example = wnut["train"][0]
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
tokens

However, this adds some special tokens `[CLS]` and `[SEP]` and the subword tokenization creates a mismatch between the input and labels. A single word corresponding to a single label may now be split into two subwords. You'll need to realign the tokens and labels by:

1. Mapping all tokens to their corresponding word with the [`word_ids`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.BatchEncoding.word_ids) method.
2. Assigning the label `-100` to the special tokens `[CLS]` and `[SEP]` so they're ignored by the PyTorch loss function (see [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)).
3. Only labeling the first token of a given word. Assign `-100` to other subtokens from the same word.

Here is how you can create a function to realign the tokens and labels, and truncate sequences to be no longer than DistilBERT's maximum input length:

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    # print(examples)
    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        # Map tokens to their respective word.
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Set the special tokens to -100.
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

To apply the preprocessing function over the entire dataset, use 🤗 Datasets [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map) function. You can speed up the `map` function by setting `batched=True` to process multiple elements of the dataset at once:

In [ ]:
tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True)

#### what happens inside the preprocess method 👇

In [ ]:
# batch = {
#     "id": wnut["train"][:3]["id"],
#     "tokens": wnut["train"][:3]["tokens"],
#     "ner_tags": wnut["train"][:3]["ner_tags"],
# }

# # print("BATCH INPUT:")
# # print(batch)

# tokenized_inputs = tokenizer(
#     batch["tokens"],
#     truncation=True,
#     is_split_into_words=True
# )

# # print("\nTOKENIZED INPUTS:")
# # print(tokenized_inputs)

# labels = []
# for i, label in enumerate(batch["ner_tags"]):
#     word_ids = tokenized_inputs.word_ids(batch_index=i)
#     # print(f"\nWord IDs for example {i}: {word_ids}")

#     previous_word_idx = None
#     label_ids = []
#     for word_idx in word_ids:
#         if word_idx is None:
#             label_ids.append(-100)
#         elif word_idx != previous_word_idx:
#             label_ids.append(label[word_idx])
#         else:
#             label_ids.append(-100)
#         previous_word_idx = word_idx

#     labels.append(label_ids)
#     # print(f"Aligned labels for example {i}: {label_ids}")
# tokenized_inputs["labels"] = labels
# # print("\nFINAL TOKENIZED INPUTS WITH LABELS:")
# # print(tokenized_inputs['labels'])

Now create a batch of examples using [DataCollatorWithPadding](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorWithPadding). It's more efficient to *dynamically pad* the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## Evaluate

For this task, load the [seqeval](https://huggingface.co/spaces/evaluate-metric/seqeval) framework (see the 🤗 Evaluate [quick tour](https://huggingface.co/docs/evaluate/a_quick_tour) to learn more about how to load and compute a metric). Seqeval actually produces several scores: precision, recall, F1, and accuracy.

In [ ]:
# !pip install seqeval
import evaluate
seqeval = evaluate.load("seqeval")
import numpy as np

labels = [label_list[i] for i in example[f"ner_tags"]]


def compute_metrics(p):
    predictions, labels = p
    # # debug ------
    # print("Inside the compute_metrics function")
    # print(f"predictions.shape: {predictions.shape}")
    # print(f"labels.shape: {labels.shape}")
    # # debug ------

    # select the predicted index (with the maximum logit) for each token
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## Train

In [ ]:
id2label = {
    0: "O",
    1: "B-corporation",
    2: "I-corporation",
    3: "B-creative-work",
    4: "I-creative-work",
    5: "B-group",
    6: "I-group",
    7: "B-location",
    8: "I-location",
    9: "B-person",
    10: "I-person",
    11: "B-product",
    12: "I-product",
}
label2id = {
    "O": 0,
    "B-corporation": 1,
    "I-corporation": 2,
    "B-creative-work": 3,
    "I-creative-work": 4,
    "B-group": 5,
    "I-group": 6,
    "B-location": 7,
    "I-location": 8,
    "B-person": 9,
    "I-person": 10,
    "B-product": 11,
    "I-product": 12,
}

from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

The warning is telling us we are randomly initializing some weights (the `classifier` layer), so the library warns us we should fine-tune this model before using it for inference, which is exactly what we are going to do.

In [ ]:
model_name = model_checkpoint.split("/")[-1]
task = "ner"
ckpt_name = f"{model_name}-finetuned-{task}"

training_args = TrainingArguments(
    output_dir=ckpt_name,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
    report_to="none" # to disable wandb login
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_wnut["train"],
    eval_dataset=tokenized_wnut["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

The `evaluate` method allows you to evaluate again on the evaluation dataset or on another dataset:

In [ ]:
trainer.evaluate()

Once training is completed, share your model to the Hub with the [push_to_hub()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.push_to_hub) method so everyone can use your model:

In [ ]:
trainer.push_to_hub()

To get the precision/recall/f1 computed for each category now that we have finished training, we can apply the same function as before on the result of the `predict` method:

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_wnut["validation"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = seqeval.compute(predictions=true_predictions, references=true_labels)
results

## Inference

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline).

In [ ]:
from transformers import pipeline
classifier = pipeline(task, model=f"{user_name}/{ckpt_name}")

text = "The Golden State Warriors are an American professional basketball team based in San Francisco."

classifier(text)

You can also manually replicate the results of the `pipeline` if you'd like

In [ ]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(text, return_tensors="pt") # return PyTorch tensors


model = AutoModelForTokenClassification.from_pretrained(f"{user_name}/{ckpt_name}")
with torch.no_grad():
    logits = model(**inputs).logits

# Get the class with the highest probability
predictions = torch.argmax(logits, dim=2)
# Use the model's `id2label` mapping to convert it to a text label:
predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]
predicted_token_class